In [1]:
#%%     Imports

%load_ext autoreload
%autoreload 2
%matplotlib inline

# Standard Imports
import tensorflow as tf
import keras_tuner as kt 
import kormos

tf.keras.backend.clear_session()
tf.keras.backend.set_floatx('float64')
tf_float = 'float64'
tf.random.set_seed(42)

import numpy as np
np_float = np.float64

import os
import datetime

# Own imports
import ContinuumMechanics as CM
import layers
import subANNs
import Outputs
import Plots
import Callbacks
import utils
import fit
import build


device = 'gpu:' + str(0) if tf.test.is_gpu_available() else 'cpu:0'

Instructions for updating:
Use `tf.config.list_physical_devices('GPU')` instead.


In [ ]:
dataset = 'Calvo_abdom'

load_model = False
modelToLoad = '.\\Results_{:}\\20251218-200838'.format(dataset)

pathToData = '.\\datasets\\{:}_data\\'.format(dataset)   

if load_model: 
    outputFolder = modelToLoad
else:
    date = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    outputFolder = '.\\Results_{:}\\'.format(dataset) + date

if not os.path.exists(outputFolder):
    os.makedirs(outputFolder)


In [ ]:


incomp = True # 
visco = True # use the viscoelastic or purely elastic version of the model

uncoupled = True # must be True, was only experementally tested
rateDependent = False # if relaxation times are strain rate dependent

numDir = 0 # number $n$ of preferred material directions $\vec{m}$ (e.g. fibers) to be learned by the model
numTens = 1  # number $R$ of generalized Maxwell models $\tilde{\tns{L}}_r$ to be learned by the model
numExtra = 0 # number $n_f$ of features which parameterize the free energy functions and relaxation times
numExtraStruc = 0 # number of features which parameterize the preferred material direction $\vec{m}_i$ and corresponding wieghts w_i^{(r)}

# for saving and loading    
custom_objects = {
    'dirModel': CM.dirModelOrtho,
    'weightModel': CM.weightModelOrtho,
    'PositiveHeNormal': subANNs.PositiveHeNormal,
    'PositiveVarianceScaling': subANNs.PositiveVarianceScaling,
    'PositiveGlorotNormal': subANNs.PositiveGlorotNormal,
    'PsiSigmaLayer': CM.PsiSigmaLayer, 
    'GradientLayer': CM.GradientLayer,
    'ScaleLayer': layers.ScaleLayer, 
    'stressUpdateLayer': CM.stressUpdateLayer,
    'SparsityRegularizer': utils.SparsityRegularizer
}

In [ ]:
#%% Hyperparameters
lr = 0.01
clipnorm = None
lambda_maxwell = 0.0 # penalty to enforce sparsity of the Prony Series

# Time scaling constants (powers of 10)
tau_min = 1
tau_max = 3

# Some activation functions
acti0 = 'tanh'
acti1 = 'sigmoid'
acti2 = 'softplus'
acti3 = 'linear'
acti4 = 'elu'
acti5 = 'squared_softplus'
acti6 = 'neg_squared_softplus'

#
EPOCHS = 2000
earlyStopPatience = 500

# maximum number N_max of Maxwell elements per generalized Maxwell model
nMaxwell = 3

# Equilibrium free energy: number of layers / neurons per layer
layer_size_psi = [8,] # last layer of shape (1,) is automatically included in the model
activations_psi = [acti2, acti2, acti2]

# Preferred material directions: number of layers / neurons per layer
layer_size_dir = [5,]
activations_dir = [acti2, acti3, acti3]

# Weights of preferred material directions: number of layers / neurons per layer
layer_size_dir = [5,] 
activations_w = [acti2, acti3, acti2]

# Relaxation times: : number of layers / neurons per layer
layer_size_tau = [8,] # last layer of shape (1,) is automatically included in the model
activations_tau = [acti2, acti2, acti2, acti2]

# Non-equilibrium free energy: number of layers / neurons per layer
layer_size_psi_a = [8,] # last layer of shape (1,) is automatically included in the model
activations_psi_a = [acti2, acti2, acti2, acti2]

In [5]:
#%% Data
        
# Training and validation data
trainDs = tf.data.experimental.load(pathToData + "ds_train_defGrad", compression='GZIP')
valDs = tf.data.experimental.load(pathToData + "ds_train_defGrad", compression='GZIP')

tf.data.experimental.save(trainDs, outputFolder + '\\ds_train_defGrad', compression='GZIP')
tf.data.experimental.save(valDs, outputFolder + '\\ds_valid_defGrad', compression='GZIP')


Instructions for updating:
Use `tf.data.Dataset.load(...)` instead.
Instructions for updating:
Use `tf.data.Dataset.save(...)` instead.


In [ ]:
#%% build the model with prescribed hyperparameters
nSteps = int(np.loadtxt(pathToData + 'n_time_steps.txt'))

if load_model == True:
    model_fit  = Outputs.loadModel(modelToLoad, 'model_1', custom_objects)
    
else:             
    model_fit, model_full = build.build_model(
        nSteps,
        numTens,
        numDir,
        nMaxwell,
        numExtra,
        numExtraStruc,
        uncoupled,
        rateDependent,
        layer_size_psi,
        activations_psi,
        layer_size_dir,
        activations_dir,
        layer_size_w,
        activations_w,
        layer_size_tau,
        activations_tau,
        layer_size_psi_a, 
        activations_psi_a,
        [tau_min, tau_max],
        lambda_maxwell,
        incomp,
        visco,
        tf_float
        )
        
    

In [9]:
#%% Fit
normalized_error = False
stochastic = False

Maxwell_monitor = None # Callbacks.NonZeroWeightsMonitor(numTens, lambda_prony)
reg_callback = Callbacks.RegularizationCallback(lambda_maxwell, numTens)

if normalized_error == True:
    loss = fit.IndividuallyNormalizedLoss()  
else:
    # loss = tf.keras.losses.MeanSquaredError()
    loss = fit.WeightedLoss()


# stochastic optimizer
if stochastic:

    optimizer = tf.keras.optimizers.Adam(learning_rate=lr, clipnorm=clipnorm, name='optimizer')
    
    #Outputs.saveOptimizerConfig(optimizer, outputFolder=outputFolder)
    #Outputs.saveModelConfig(model_fit, outputFolder=outputFolder) 
    
    # tf.keras.backend.set_value(model_fit.optimizer.learning_rate, 0.1)
    model_fit.compile(
            optimizer=optimizer,
            loss=loss,
            run_eagerly=False
        )
    
    with tf.device('/device:CPU:0'):
        history, model_fit =  fit.stochastic(model_fit, trainDs, EPOCHS, earlyStopPatience, outputFolder, valDs=valDs, Maxwell_monitor=Maxwell_monitor)
                                    
# deterministic optimizer
else:
    with tf.device('/device:CPU:0'):
        history, model_fit = fit.deterministic(model_fit, trainDs, EPOCHS, earlyStopPatience, outputFolder, valDs=valDs, Maxwell_monitor=Maxwell_monitor, loss=loss)


#%%    
Outputs.saveLoss(history, Maxwell_monitor=Maxwell_monitor, outputFolder=outputFolder)
Outputs.plotLoss(history, Maxwell_monitor=Maxwell_monitor, title='$\Lambda = {:}$'.format(lambda_maxwell), outputFolder=outputFolder,scale='log')

Constrained model: 

   model_psi_a_0
      Psi_a_1_1_1:
         kernel constrained
         no bias constraint
      Psi_a_1_2_1:
         kernel constrained
         no bias constraint
      Psi_a_1_3_1:
         kernel constrained
         no bias constraint
      Psi_a_1_1_2:
         kernel constrained
      Psi_a_1_2_2:
         kernel constrained
      Psi_a_1_3_2:
         kernel constrained
   model_Psi
      Psi_1_1:
         kernel constrained
         no bias constraint
      Psi_1_2:
         kernel constrained
   model_tau_0
      Tau_1_1_1:
         no kernel constraint
         no bias constraint
      Tau_1_2_1:
         no kernel constraint
         no bias constraint
      Tau_1_3_1:
         no kernel constraint
         no bias constraint
      Tau_1_1_2:
         no kernel constraint
         no bias constraint
      Tau_1_2_2:
         no kernel constraint
         no bias constraint
      Tau_1_3_2:
         no kernel constraint
         no bias constraint
Tota

In [ ]:
#%% Plot graphs and save model (summary)

Outputs.showModelSummary(model_fit, outputFolder)
Outputs.plotModelGraph(model_fit, outputFolder)
Outputs.saveModel(model_fit, outputFolder)





Model: "model_31"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 F_input (InputLayer)           [(None, 1000, 3, 3)  0           []                               
                                ]                                                                 
                                                                                                  
 tf.compat.v1.shape_5 (TFOpLamb  (4,)                0           ['F_input[0][0]']                
 da)                                                                                              
                                                                                                  
 tf.__operators__.getitem_180 (  ()                  0           ['tf.compat.v1.shape_5[0][0]']   
 SlicingOpLambda)                                                                      

In [76]:
res = model_full.predict(trainDs)
n=1

S_infty = res[16]
Tau = res[17]
S = res[18][0,n] 
# S_neq = res[39]
dPsi_a_dI_ref = res[25][1]
dPsi_a_dJ_ref = res[26][2]
alpha_a = res[28]
beta_a = res[29]
S_a = res[32]
Q_unstack = res[33]
P = res[34]
# dPsi_a_dI_c = res[-9]
# dPsi_a_dJ_c = res[-8]
# ddPsi_a_ddI_c = res[-7]
# ddPsi_a_ddJ_c = res[-6]
# ddPsi_a_dIdJ_c = res[-5]
# S_ra_neq_bar = res[38]
# dSdCbar = res[35]
# dS_a_dC_bar = res[36]
# dS_a_neq_bar_1, dS_a_neq_bar_2, dS_a_neq_bar_3 = res[-3], res[-2], res[-1]

# Q_unstack[2][0,]
# time[0,n]
#S_a[0,n]
#Tau[0,n]
# dPsi_a_dI_c[0,n]
# print(S_ra_neq_bar[0,n,0])
# print(dS_a_neq_bar_1[n][0,n])
# print(dS_a_neq_bar_2[n][0,n])
# print(dS_a_neq_bar_3[n][0,n])
# Tau[0,n]
# print((dS_a_dC_bar[0][0,n,0,0,:,:] + dS_a_dC_bar[0][0,n,0,0,:,:] + dS_a_dC_bar[0][0,n,0,0,:,:])*2)

# # dQdCbar_2[0][0,0]
# print(dS_a_neq_bar_3[n][0,n,2,2])





1/1 [==============================] - 11s 11s/step


In [77]:
Tau

[array([[[4.40776329e+00, 4.17656850e+01, 5.77748302e+03],
         [4.40776329e+00, 4.17656850e+01, 5.77748302e+03],
         [4.40776329e+00, 4.17656850e+01, 5.77748302e+03],
         ...,
         [4.40776329e+00, 4.17656850e+01, 5.77748302e+03],
         [4.40776329e+00, 4.17656850e+01, 5.77748302e+03],
         [4.40776329e+00, 4.17656850e+01, 5.77748302e+03]],
 
        [[4.27821693e+00, 4.15724593e+01, 6.01324528e+03],
         [4.27821693e+00, 4.15724593e+01, 6.01324528e+03],
         [4.27821693e+00, 4.15724593e+01, 6.01324528e+03],
         ...,
         [4.27821693e+00, 4.15724593e+01, 6.01324528e+03],
         [4.27821693e+00, 4.15724593e+01, 6.01324528e+03],
         [4.27821693e+00, 4.15724593e+01, 6.01324528e+03]],
 
        [[4.15420834e+00, 4.13354183e+01, 6.28199812e+03],
         [4.15420834e+00, 4.13354183e+01, 6.28199812e+03],
         [4.15420834e+00, 4.13354183e+01, 6.28199812e+03],
         ...,
         [4.15420834e+00, 4.13354183e+01, 6.28199812e+03],
        

In [10]:
%matplotlib qt

#%% Stress Response #
Plots.plot_calvo(model_fit, trainDs, valDs, outputFolder)

1/1 [==============================] - 0s 98ms/step


In [ ]:
from checkpoint_analysis import parallel_checkpoint_analysis
parallel_checkpoint_analysis(model_fit, pathToData, outputFolder, rateDependent, checkpoint_dir=None, max_workers=None)

In [ ]:
from checkpoint_analysis import create_video_from_checkpoint_analysis
video_folder = "C:\\Users\\Kian\\Documents\\Projekte\\ViscoCANN\\vCANN\\Results_General_biaxial\\20251125-015147"
create_video_from_checkpoint_analysis(video_folder, fps=2, duration_per_frame=0.5)